# Bronze Layer Ingestion Pipeline — Serie A Player Statistics
### Databricks Lakehouse | API-Football | Medallion Architecture

This notebook implements the full Bronze layer of a production-grade data pipeline. It extracts raw player statistics for the 2024 Serie A season from the API-Football REST API and lands them, untransformed, into a governed Delta Lake table within Unity Catalog.

**Architecture (5-Cell Structure):**
- **Cell 1:** Pipeline Configuration
- **Cell 2:** Pre-Flight Metadata Validation
- **Cell 3:** Unity Catalog Infrastructure Setup
- **Cell 4:** Distributed Extraction Engine
- **Cell 5:** Auto Loader Streaming & Delta Lake Commit

**Output:** `bronze_dev.api_sports.raw_serie_a_players_2024` (Delta format, Unity Catalog governed)

## Cell 1: Pipeline Configuration
Modify only this cell to target a different league, country, or season. All downstream cells inherit these variables automatically.

In [0]:
import requests
import time
import json

# --- RUNTIME PARAMETERS ---
# PRODUCTION PATTERN: API key should be retrieved via Databricks Secrets:
# API_KEY = dbutils.secrets.get(scope="portfolio-pipeline", key="api-football-key")
# Free Edition does not support Secrets — replace the value below before running.
API_KEY = "3beee6a9634f5da2e02ff8a9425781c8"

TARGET_LEAGUE_NAME = "Serie A"
TARGET_COUNTRY = "Italy"
TARGET_SEASON = "2024"

# --- ENDPOINTS ---
HEADERS = {"x-apisports-key": API_KEY}
URL_LEAGUES = "https://v3.football.api-sports.io/leagues"
URL_TEAMS = "https://v3.football.api-sports.io/teams"
URL_PLAYERS = "https://v3.football.api-sports.io/players"

print(f"Pipeline configured for: {TARGET_LEAGUE_NAME}, {TARGET_COUNTRY} ({TARGET_SEASON})")

## Cell 2: Pre-Flight Metadata Validation
Validates all runtime parameters against the source API before consuming any daily quota. Raises a descriptive error immediately if a parameter is invalid, preventing cascading downstream failures.

In [0]:
print("Validating parameters with source system...")

# 1. Validate League and Country
league_params = {"name": TARGET_LEAGUE_NAME, "country": TARGET_COUNTRY}
league_response = requests.get(URL_LEAGUES, headers=HEADERS, params=league_params).json()

# FAULT TOLERANCE: Did the API return an empty list because of a typo?
if not league_response.get("response"):
    raise ValueError(f"CRITICAL ERROR: Could not find '{TARGET_LEAGUE_NAME}' in '{TARGET_COUNTRY}'. Please check your spelling in Cell 1.")

league_id = league_response["response"][0]["league"]["id"]
print(f"League Validated! Internal ID: {league_id}")

# 2. Validate Season and Teams
team_params = {"league": league_id, "season": TARGET_SEASON}
teams_response = requests.get(URL_TEAMS, headers=HEADERS, params=team_params).json()
team_ids = [item["team"]["id"] for item in teams_response.get("response", [])]

# FAULT TOLERANCE: Did the API return 0 teams because the season year is invalid/future?
if not team_ids:
    raise ValueError(f"CRITICAL ERROR: No teams found for the {TARGET_SEASON} season. Verify the year.")

print(f"Season Validated! Located {len(team_ids)} teams.")

## Cell 3: Unity Catalog Infrastructure Setup
Ensures the landing zone folder exists and preserves all previously landed files, allowing Auto Loader's checkpoint to correctly identify new files on every subsequent run.

In [0]:
clean_league_name = TARGET_LEAGUE_NAME.lower().replace(" ", "_")

catalog_name = "bronze_dev"
schema_name = "api_sports"
table_name = f"raw_{clean_league_name}_players_{TARGET_SEASON}"
volume_name = "landing_zone"

volume_folder_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{clean_league_name}_{TARGET_SEASON}/"
checkpoint_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/_checkpoints/{clean_league_name}_{TARGET_SEASON}/"
full_table_path = f"{catalog_name}.{schema_name}.{table_name}"

print("Provisioning Unity Catalog Lakehouse Infrastructure...")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

dbutils.fs.mkdirs(volume_folder_path)  # Creates folder only if it doesn't exist

print(f"Infrastructure Ready. Landing zone folder secured at: {volume_folder_path}")

## Cell 4: Distributed Extraction Engine
Iterates through all validated Team IDs, handles dynamic pagination, and writes each API response as its own uniquely timestamped JSON file directly to the Unity Catalog Volume, ensuring every run produces new files that Auto Loader can incrementally detect. Enforces a 7-second delay between requests to respect the API's rate limit, and distinguishes between hard API limit errors (clean exit) and transient errors (retry).

In [0]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Starting Direct-to-Disk Extraction to Volume Folder...")

for team_id in team_ids:
    current_page = 1
    total_pages_for_team = 1

    while current_page <= total_pages_for_team:
        player_params = {"team": team_id, "season": TARGET_SEASON, "page": current_page}
        response = requests.get(URL_PLAYERS, headers=HEADERS, params=player_params).json()

        # FAULT TOLERANCE: Distinguish between a hard limit vs. a transient error
        if response.get("errors"):
            error_msg = str(response["errors"])
            if "Page parameter" in error_msg:
                # Hard API limit hit — no point retrying, just move to next team
                print(f"  -> Page limit reached for team {team_id} at page {current_page}. Moving on.")
                break
            else:
                # Transient error (rate limit, network blip) — retry same page
                print(f"  -> Transient error on team {team_id} page {current_page}: {error_msg}. Retrying in 15s...")
                time.sleep(15)
                continue

        if current_page == 1 and "paging" in response:
            total_pages_for_team = response["paging"]["total"]

        if response.get("response"):
            file_path = f"{volume_folder_path}team_{team_id}_page_{current_page}_{timestamp}.json"

            # Unity Catalog Volumes are POSIX-accessible, so standard Python I/O works natively
            with open(file_path, "w") as f:
                for player_record in response["response"]:
                    f.write(json.dumps(player_record) + "\n")

            print(f"  -> Streamed File: team_{team_id}_page_{current_page}.json")

        current_page += 1
        time.sleep(7)

print("Extraction Complete! Distributed files are resting safely in the Volume folder.")

## Cell 5: Auto Loader Streaming & Delta Lake Commit
Incrementally loads all extracted JSON files into the Delta table using Databricks Auto Loader. Handles nested schema inference, dynamic schema evolution, and fault tolerance automatically. Uses `availableNow` trigger for cost efficiency and `awaitTermination()` to block until the batch is fully committed.

In [0]:
print(f"Initializing Databricks Auto Loader with Schema Evolution...")

# Variables (TARGET_SEASON, volume_folder_path, etc.) are seamlessly inherited from Cells 1 and 3.

# --- READ STREAM ---
df_raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(volume_folder_path)
)

# --- WRITE STREAM ---
query = (
    df_raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    # THE MAGIC BULLET: Allow the Delta table schema to mutate and evolve dynamically
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(full_table_path)
)

# BLOCKING: Force the notebook to wait until the asynchronous stream actually finishes
query.awaitTermination()

print(f"AUTO LOADER SUCCESS! Fault-tolerant stream completed to: {full_table_path}")

## Bronze Layer Complete

The pipeline successfully extracted and landed **1,126 raw player records** across all 20 Serie A clubs for the 2024 season.

**Output Delta Table:** `bronze_dev.api_sports.raw_serie_a_players_2024`  
**Storage:** Unity Catalog Volume → Delta Lake (ACID compliant, schema evolution enabled)  
**Data State:** Raw, untransformed — exact API payload preserved for full auditability.

The next stage is the **Silver Layer**, where the deeply nested JSON structs will be flattened, Colombian players will be filtered, and the data will be joined and cleaned into an analytics-ready table.